# 09 · Larger-file strategies and Parquet

**Module:** 2 · **Time:** 40 minutes · **Prerequisites:** DataFrames, types, and reproducibility basics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/09_larger_data.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F09_larger_data.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** These badges link to the currently private repository's `main` after a future merge.

## Learning goals

- **I**nspect the memory cost of a growing DataFrame.
- **R**ead CSV in chunks and select only necessary variables.
- **C**ompare efficient categorical types and Parquet storage.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Synthetic teaching extract:", survey.shape)

## Why this matters

A beginner may be able to load a tiny CSV but struggle with a multi-gigabyte research file. Start with memory inspection and selective reads before adopting extra tools. This lesson repeats fictional records to show *software behavior*, not a larger representative sample.

## Expand synthetic rows for a memory demonstration

The repeated records are not independent observations. They simply give us enough rows to see how type choices affect memory.

In [ ]:
large = pd.concat([survey.assign(batch=i) for i in range(150)], ignore_index=True)
print("Rows for performance practice:", len(large))
print("Approximate bytes:", int(large.memory_usage(deep=True).sum()))

## Use categorical types for repeated labels

Repeated text labels are often cheaper to store as categoricals. Always inspect whether the conversion changes meaning or category ordering.

In [ ]:
compact = large.astype({"region": "category", "employment": "category"})
before = int(large.memory_usage(deep=True).sum())
after = int(compact.memory_usage(deep=True).sum())
print("Bytes before / after:", before, after)

## Read only required CSV columns and chunks

`usecols` reduces imported columns, while `chunksize` returns an iterator. Summing chunk counts should recover the number of CSV rows.

In [ ]:
from io import StringIO

csv_text = large.to_csv(index=False)
reader = pd.read_csv(StringIO(csv_text), chunksize=200, usecols=["person_id", "survey_weight"])
chunk_count = sum(len(chunk) for chunk in reader)
assert chunk_count == len(large)
print("Rows across chunks:", chunk_count)

## Parquet on the installed engine

Parquet preserves data types and is efficient for repeated reads. The locked course environment includes pyarrow; the local sandbox may not, so the optional demo checks availability.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    if find_spec("pyarrow") is not None:
        path = Path(directory) / "survey.parquet"
        compact.to_parquet(path, index=False)
        restored = pd.read_parquet(path)
        print("Parquet rows:", len(restored))
    else:
        print("Parquet needs pyarrow; run uv sync --locked.")

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Read columns | `pd.read_csv(..., usecols=...)` | `import delimited` then `keep` | `readr::read_csv(col_select=...)` |
| Count memory | `df.memory_usage(deep=True)` | `describe` / `memory` | `object.size(df)` |
| Read chunks | `pd.read_csv(..., chunksize=...)` | file-by-file loops | `readr::read_delim_chunked(...)` |
| Parquet | `pd.read_parquet(path)` | via supported imports/packages | `arrow::read_parquet(path)` |

When memory still does not fit, consult documented tools such as Polars or DuckDB; they are not dependencies of this core lesson.

## Try it

Fill each answer cell. In the learner edition, incomplete answers print hints without stopping notebook execution.

### ★ Exercise 1 — Inspect the DataFrame memory

Set `memory_bytes` to an integer giving `large`'s memory usage including object contents.

In [ ]:
memory_bytes = None

In [ ]:
expected = int(large.memory_usage(deep=True).sum())
if isinstance(memory_bytes, (int, np.integer)) and memory_bytes == expected:
    print("✓ Memory measurement correct")
else:
    print("Hint: call memory_usage(deep=True).sum(), then int(...).")

### ★★ Exercise 2 — Convert text to categorical

Create `region_categories` as a categorical Series from `large['region']`.

In [ ]:
region_categories = None

In [ ]:
expected = large["region"].astype("category")
if isinstance(region_categories, pd.Series) and region_categories.equals(expected):
    print("✓ Categorical conversion correct")
else:
    print("Hint: use astype('category') on the region column.")

### ★★ Exercise 3 — Read a subset of columns

Create `small_columns` by reading just `person_id` and `survey_weight` from `csv_text`.

In [ ]:
small_columns = None

In [ ]:
expected = pd.read_csv(StringIO(csv_text), usecols=["person_id", "survey_weight"])
if isinstance(small_columns, pd.DataFrame) and small_columns.equals(expected):
    print("✓ Selective CSV import correct")
else:
    print("Hint: pd.read_csv accepts a usecols list.")

### ★★★ Exercise 4 — Read a CSV using chunks

Calculate `chunk_total` as the number of rows returned across chunked imports of csv_text, with chunksize 200.

In [ ]:
chunk_total = None

In [ ]:
expected = len(large)
if isinstance(chunk_total, (int, np.integer)) and chunk_total == expected:
    print("✓ Chunk count correct")
else:
    print("Hint: sum the number of rows in each chunk.")

## Common errors

- Treating repeated example rows as independent data produces invalid statistical claims; this example is for memory testing only.
- Reading every column wastes memory when only a subset is needed.
- Saving Parquet requires a supported engine such as pyarrow; check the project environment.
- Changing types without checking missingness and category meaning can silently corrupt analysis.

## Recap

- Inspect deep memory usage and carefully choose dtypes.
- Use selective CSV reads and chunks before adding infrastructure.
- Parquet stores column types and is useful for repeated analysis.
- Scaling tools improve computation, not the representativeness of synthetic samples.

## Next steps

Next: [Module 2 wrangling project](10_wrangling_project.ipynb) (planned). Reference: [pandas I/O documentation](https://pandas.pydata.org/docs/user_guide/io.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).